# AirBnB Rental Price Analysis

This notebook explores the AirBnB dataset to understand what factors influence rental prices in New York City. We will perform Exploratory Data Analysis (EDA), preprocess the data, and build a predictive model for rental prices.

**Target Variable:** `price`  
**Goal:** Understand the key drivers of AirBnB pricing and build a model to predict rental prices.

## 1. Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set a clean style for all plots
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

## 2. Loading the Data

In [ ]:
df = pd.read_csv('dataSP23.csv')
df.head()

---
## 3. Exploratory Data Analysis (EDA)

Before any preprocessing or modelling, we explore the dataset to understand its structure, distributions, and relationships.

### 3.1 Dataset Overview

In [ ]:
print('Shape of dataset:', df.shape)
print('\nColumn Data Types:')
print(df.dtypes)

**Observations:**
- The dataset has **27,379 rows** and **16 columns**.
- Columns like `neighbourhood_group` and `room_type` are categorical (object type).
- `last_review` is a date stored as a string — it will be dropped as it is not directly useful for price prediction.

### 3.2 Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print('Columns with Missing Values:')
print(missing_df if not missing_df.empty else 'No missing values found!')

In [ ]:
# Visual heatmap of missing values
plt.figure(figsize=(12, 4))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Missing Values Heatmap (Yellow = Missing)', fontsize=14)
plt.tight_layout()
plt.show()

**Observation:** Columns `last_review` and `reviews_per_month` have missing values. These will be dropped in preprocessing.

### 3.3 Summary Statistics

In [ ]:
df.describe()

**Key Observations:**
- The **average price** is ~\$151, but the **max is \$10,000**, indicating heavy right skew and extreme outliers.
- The **median price** (50th percentile) is \$105, which is much lower than the mean — confirming skewness.
- `minimum_nights` has a max of 999, which is likely a data entry error.

### 3.4 Price Distribution (Target Variable)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw price distribution (capped at 1000 for visibility)
price_capped = df[df['price'] <= 1000]['price']
axes[0].hist(price_capped, bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Price Distribution (capped at $1000)', fontsize=13)
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Count')
axes[0].axvline(df['price'].median(), color='red', linestyle='--', label=f'Median: ${df["price"].median()}')
axes[0].axvline(df['price'].mean(), color='orange', linestyle='--', label=f'Mean: ${df["price"].mean():.0f}')
axes[0].legend()

# Log-transformed price (more normal distribution for modeling)
log_price = np.log1p(df[df['price'] > 0]['price'])
axes[1].hist(log_price, bins=50, color='teal', edgecolor='white')
axes[1].set_title('Log-Transformed Price Distribution', fontsize=13)
axes[1].set_xlabel('log(Price + 1)')
axes[1].set_ylabel('Count')

plt.suptitle('Distribution of AirBnB Rental Prices', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

**Observations:**
- The raw price distribution is **heavily right-skewed** — most listings are affordable, but a few luxury listings drive the mean up.
- After a **log transformation**, the distribution becomes much more symmetric (bell-shaped), which is better for linear models.
- The red dashed line (median ~\$105) is lower than the orange dashed line (mean ~\$151), confirming the skewness caused by high-price outliers.

### 3.5 Categorical Feature Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Neighbourhood Group
nbr_counts = df['neighbourhood_group'].value_counts()
axes[0].bar(nbr_counts.index, nbr_counts.values, color=sns.color_palette('muted', len(nbr_counts)))
axes[0].set_title('Listings by Neighbourhood Group', fontsize=13)
axes[0].set_xlabel('Neighbourhood Group')
axes[0].set_ylabel('Number of Listings')
for i, v in enumerate(nbr_counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', fontsize=10)

# Room Type
room_counts = df['room_type'].value_counts()
axes[1].bar(room_counts.index, room_counts.values, color=sns.color_palette('Set2', len(room_counts)))
axes[1].set_title('Listings by Room Type', fontsize=13)
axes[1].set_xlabel('Room Type')
axes[1].set_ylabel('Number of Listings')
for i, v in enumerate(room_counts.values):
    axes[1].text(i, v + 100, str(v), ha='center', fontsize=10)

plt.suptitle('Categorical Feature Distributions', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

**Observations:**
- **Manhattan** and **Brooklyn** dominate the listings, together accounting for the majority of AirBnB rentals in NYC.
- **Entire home/apt** and **Private room** are by far the most common room types. **Shared rooms** are a small fraction of listings.

### 3.6 Price by Neighbourhood Group & Room Type

In [ ]:
# Filter to remove extreme outliers for cleaner visualization
df_filtered = df[df['price'] <= 500]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Price by Neighbourhood Group
order_nbr = df_filtered.groupby('neighbourhood_group')['price'].median().sort_values(ascending=False).index
sns.boxplot(data=df_filtered, x='neighbourhood_group', y='price',
            order=order_nbr, palette='muted', ax=axes[0])
axes[0].set_title('Price by Neighbourhood Group', fontsize=13)
axes[0].set_xlabel('Neighbourhood Group')
axes[0].set_ylabel('Price ($)')
axes[0].tick_params(axis='x', rotation=15)

# Price by Room Type
order_room = df_filtered.groupby('room_type')['price'].median().sort_values(ascending=False).index
sns.boxplot(data=df_filtered, x='room_type', y='price',
            order=order_room, palette='Set2', ax=axes[1])
axes[1].set_title('Price by Room Type', fontsize=13)
axes[1].set_xlabel('Room Type')
axes[1].set_ylabel('Price ($)')

plt.suptitle('Price Distribution by Category (capped at $500)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

**Key Insights:**
- **Manhattan** has the highest median price, significantly above other boroughs. **The Bronx** and **Staten Island** are the most affordable.
- **Entire home/apt** listings cost significantly more than **private rooms**, which in turn cost more than **shared rooms**.
- These two features (`neighbourhood_group` and `room_type`) are strong predictors of price.

### 3.7 Correlation Heatmap

In [ ]:
# Select only numerical columns
num_cols = ['price', 'minimum_nights', 'number_of_reviews',
            'reviews_per_month', 'calculated_host_listings_count', 'availability_365']

corr_matrix = df[num_cols].corr()

plt.figure(figsize=(9, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # show only lower triangle
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, square=True, linewidths=0.5, vmin=-1, vmax=1)
plt.title('Correlation Heatmap of Numerical Features', fontsize=14)
plt.tight_layout()
plt.show()

**Observations:**
- `number_of_reviews` and `reviews_per_month` are positively correlated — as expected.
- `price` has **low correlation** with most numerical features, suggesting the categorical features (`room_type`, `neighbourhood_group`) are more informative.
- `minimum_nights` has a slight negative correlation with `number_of_reviews` — longer minimum stay requirements may discourage bookings.

### 3.8 Geographic Distribution of Listings

In [ ]:
# Cap price at 500 for better color scale visibility
df_geo = df[df['price'] <= 500].copy()

plt.figure(figsize=(12, 8))
scatter = plt.scatter(
    df_geo['longitude'], df_geo['latitude'],
    c=df_geo['price'], cmap='plasma', alpha=0.4, s=5
)
plt.colorbar(scatter, label='Price ($)')
plt.title('Geographic Distribution of AirBnB Listings\n(Color = Price, capped at $500)', fontsize=14)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.tight_layout()
plt.show()

**Observations:**
- **Manhattan** (center-top cluster) clearly shows **higher prices** (brighter colors).
- **Brooklyn** (lower-left cluster) shows a mix of mid-range prices.
- **Outer boroughs** (Queens, Bronx, Staten Island) have a higher concentration of lower-priced listings (darker colors).
- This confirms that **geographic location is a major driver** of AirBnB pricing in NYC.

### 3.9 Average Price by Neighbourhood Group (Bar Chart)

In [ ]:
avg_price = df.groupby('neighbourhood_group')['price'].mean().sort_values(ascending=False)

plt.figure(figsize=(9, 5))
bars = plt.bar(avg_price.index, avg_price.values,
               color=sns.color_palette('coolwarm', len(avg_price)))
plt.title('Average Price by Neighbourhood Group', fontsize=14)
plt.xlabel('Neighbourhood Group')
plt.ylabel('Average Price ($)')
for bar, val in zip(bars, avg_price.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             f'${val:.0f}', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

**Observations:**
- **Manhattan** has the highest average price, nearly double that of The Bronx.
- **Staten Island** has a relatively high average — this is likely pulled upward by a few expensive listings.
- **Queens and The Bronx** are the most budget-friendly boroughs for AirBnB guests.

### 3.10 EDA Summary & Key Insights

| Finding | Detail |
|---|---|
| **Price is right-skewed** | Most listings are affordable; a few luxury listings create outliers |
| **Manhattan is the most expensive** | Highest median and average price across all boroughs |
| **Entire home/apt costs the most** | Significantly higher prices than private or shared rooms |
| **No missing values after dropping** | `last_review` and `reviews_per_month` had NaNs — safely dropped |
| **Weak numerical correlations** | Categorical features are more predictive of price |
| **Geographic clustering is visible** | Latitude/longitude clearly separates price tiers |


---
## 4. Data Preprocessing

Now that we understand the data, we drop columns that are not useful for price prediction. These include identifier columns (`id`, `name`, `host_id`, `host_name`), date columns (`last_review`), and review-based columns that are indirectly related or have missing values.

In [ ]:
df = df.drop(['id', 'name', 'host_id', 'host_name', 'last_review',
              'number_of_reviews', 'reviews_per_month', 'neighbourhood'], axis=1)
df.head()

In [ ]:
# Confirm no missing values remain
print('Missing values after dropping columns:')
print(df.isnull().sum())

In [ ]:
# Check for duplicate rows
print('Duplicate rows:', df.duplicated().sum())

In [ ]:
print('Dataset shape after preprocessing:', df.shape)

### 4.1 Outlier Detection

In [ ]:
def check_outliers(df, col):
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f'{col}: {len(outliers)} outliers ({len(outliers)/len(df)*100:.1f}% of data) | Range: [{lower_bound:.1f}, {upper_bound:.1f}]')
    return outliers

print('Outlier Summary:')
for col in ['price', 'minimum_nights', 'calculated_host_listings_count', 'availability_365']:
    check_outliers(df, col)

**Observation:** The `price` column has ~6% outliers (very high-priced listings). These outliers significantly affect model accuracy, which explains the low R² score we'll see later.

## 5. Label Encoding

Machine learning models require numerical input. We encode the categorical columns `neighbourhood_group` and `room_type` using Label Encoding.

In [ ]:
print('Unique values in room_type:', df['room_type'].unique())
print('Unique values in neighbourhood_group:', df['neighbourhood_group'].unique())

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['neighbourhood_group'] = le.fit_transform(df['neighbourhood_group'])
df['room_type'] = le.fit_transform(df['room_type'])

df.head()

## 6. Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('price', axis=1)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_safe, y_train, y_safe = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

print(f'Training set size:   {X_train.shape[0]} samples')
print(f'Test set size:       {X_test.shape[0]} samples')
print(f'Hold-out (safe) set: {X_safe.shape[0]} samples')

## 7. Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()

X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

## 8. Model Training — Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression

regressor = LinearRegression()
regressor.fit(X_train, y_train)

## 9. Predictions & Model Evaluation

In [ ]:
from sklearn.metrics import r2_score

y_pred = regressor.predict(X_test)

r2 = r2_score(y_test, y_pred)
print(f'R² Score: {r2:.4f}')
print(f'Model Accuracy on Test Set: {100 * regressor.score(X_test, y_test):.2f}%')

**Why is the R² score low?**

The linear regression model has a low R² (~14%) due to several reasons:
1. **Outliers:** The price column contains extreme values (up to $10,000) that the linear model cannot fit well.
2. **Non-linear relationships:** Price may not have a simple linear relationship with the features.
3. **Label encoding limitation:** Assigning arbitrary numbers to `neighbourhood_group` implies an ordering that doesn't exist.

Potential improvements: remove outliers, use log-transformed price, try tree-based models (Random Forest, XGBoost), or use one-hot encoding for categorical variables.

## 10. Conclusions

This notebook explored the NYC AirBnB dataset and built a baseline price prediction model.

**Key Takeaways:**
- AirBnB prices are **heavily right-skewed** with many luxury outliers.
- **Neighbourhood group** and **room type** are the strongest predictors of price.
- **Manhattan** is the most expensive borough; **Entire home/apt** commands the highest prices.
- A simple **Linear Regression** model achieves ~14% R², limited by outliers and non-linearity.
- Future work should explore tree-based models and outlier removal for better predictions.

**Beneficiaries of this analysis:**
- **Hosts** can use insights to set competitive prices.
- **Guests** can identify the best-value listings by neighbourhood and room type.
- **Investors & city planners** can understand housing market dynamics across NYC boroughs.